In [1]:
# Natural Language ToolKit
import nltk

# nltk.download("wordnet")
# nltk.download("omw-1.4")

In [2]:
# !pip install bert-score
# !pip install rouge-score

In [3]:
from bert_score import score as bert_score
from rouge_score import rouge_scorer
from nltk.translate.meteor_score import meteor_score
import pandas as pd

# LOAD DATA
df = pd.read_csv("ai_teacher_results.csv")

# Preprocess (Convert the Teacher Response and GeneratedAnswer columns into Python lists of strings, 
# replacing missing values (NaN) with empty strings ("")).
references = (df["Teacher Response"].fillna("").astype(str).tolist())

candidates = (df["GeneratedAnswer"].fillna("").astype(str).tolist())

# BERTSCORE
p, R, F1 = bert_score(candidates, references, lang="en", verbose=True)

df["BERT"] = F1.cpu().numpy()

# ROUGE
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for candidate, reference in zip(candidates, references):
    scores = rouge.score(reference, candidate)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

df["ROUGE-1"] = rouge1_scores
df["ROUGE-2"] = rouge2_scores
df["ROUGE-L"] = rougeL_scores

# METEOR
meteor_scores = []

for candidate, reference in zip(candidates, references):
    candidate_tokens = candidate.split()
    reference_tokens = reference.split()
    score = meteor_score([reference_tokens], candidate_tokens)
    meteor_scores.append(score)

df["METEOR"] = meteor_scores

# ROUND SCORES

similarity_columns = ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]

df[similarity_columns] = (df[similarity_columns].round(3))

# QUESTION-WISE SIMILARITY DATAFRAME
similarity_result = df[["User Query", "BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"]].copy()

# DISPLAY QUESTION-WISE DATAFRAME
display(similarity_result)

# CALCULATE AVERAGES
average_bert = df["BERT"].mean()
average_rouge1 = df["ROUGE-1"].mean()
average_rouge2 = df["ROUGE-2"].mean()
average_rougeL = df["ROUGE-L"].mean()
average_meteor = df["METEOR"].mean()

# CREATE AVERAGE SIMILARITY DATAFRAME

average_similarity = pd.DataFrame({"Metric": ["BERT", "ROUGE-1", "ROUGE-2", "ROUGE-L", "METEOR"],
                                   "Average Score": [average_bert, average_rouge1, average_rouge2, average_rougeL, average_meteor]})

# ROUND AVERAGE SCORES
average_similarity["Average Score"] = (average_similarity["Average Score"].round(3))

# DISPLAY AVERAGE DATAFRAME
display(average_similarity)

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

[transformers] RobertaModel LOAD REPORT from: roberta-large
Key                       | Status     | 
--------------------------+------------+-
lm_head.dense.weight      | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.bias              | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


calculating scores...
computing bert embedding.


  0%|          | 0/1 [00:00<?, ?it/s]

computing greedy matching.


  0%|          | 0/1 [00:00<?, ?it/s]

done in 9.67 seconds, 2.07 sentences/sec


,User Query,BERT,ROUGE-1,ROUGE-2,ROUGE-L,METEOR
0,What is AI?,0.908,0.407,0.115,0.333,0.169
1,How does AI differ from traditional programming?,0.882,0.308,0.032,0.215,0.187
2,What is Machine Learning?,0.917,0.488,0.175,0.293,0.459
3,What is Deep Learning?,0.918,0.500,0.244,0.333,0.605
4,"What is the difference between AI, ML, and Dee...",0.885,0.462,0.209,0.376,0.499
5,What is training data?,0.909,0.506,0.182,0.405,0.434
6,What is a Machine Learning model?,0.912,0.416,0.160,0.390,0.342
7,What is supervised learning?,0.865,0.200,0.029,0.171,0.153
8,What is unsupervised learning?,0.892,0.282,0.058,0.197,0.197
9,What is reinforcement learning?,0.899,0.222,0.086,0.194,0.202


,Metric,Average Score
0,BERT,0.893
1,ROUGE-1,0.335
2,ROUGE-2,0.115
3,ROUGE-L,0.262
4,METEOR,0.297


In [4]:
df.to_csv("ai_teacher_evaluation_Similarity_metrics.csv", index = False, encoding = "utf-8-sig")

In [5]:
average_similarity.to_csv("ai_teacher_evaluation_Similarity_Average.csv", index = False, encoding = "utf-8-sig")

In [6]:
# nltk: 3.10.0
# bert-score: 0.3.13
# rouge-score: 0.1.2
# pandas: 2.2.2